# RAW Event NPZ Explorer

Use `pixi run raw-to-npz data` first to create compressed event arrays, then run the cells below.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path | None = None) -> Path:
    path = (Path.cwd() if start is None else start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pixi.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find repo root containing pixi.toml")


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "data"
DATA_ROOT

In [ ]:
def find_event_arrays(data_root: Path = DATA_ROOT) -> list[Path]:
    if not data_root.exists():
        return []
    return sorted(data_root.rglob("*_events.npz"))


array_files = find_event_arrays()
if not array_files:
    print(f"No *_events.npz files found under {DATA_ROOT}")
    print("Create them with: pixi run raw-to-npz data")
else:
    file_picker = widgets.Dropdown(
        options=[(str(path.relative_to(DATA_ROOT)), path) for path in array_files],
        description="Array",
        layout=widgets.Layout(width="100%"),
    )
    display(file_picker)

In [ ]:
def selected_array_path() -> Path:
    if "file_picker" not in globals():
        raise RuntimeError("Run the file picker cell after creating *_events.npz files.")
    return Path(file_picker.value)


def load_event_npz(path: Path) -> tuple[np.ndarray, tuple[int, int], dict[str, object]]:
    with np.load(path, allow_pickle=False) as archive:
        events = archive["events"]
        sensor_shape = tuple(int(value) for value in archive["sensor_shape"])
        metadata = json.loads(str(archive["metadata"]))
    return events, sensor_shape, metadata


events, sensor_shape, metadata = load_event_npz(selected_array_path())
print(f"Loaded: {selected_array_path().relative_to(DATA_ROOT)}")
print(f"Events: {events.size:,}")
print(f"Sensor shape: {sensor_shape} (image[y, x])")
print(f"Dtype: {events.dtype}")
print(f"Time range: {metadata.get('time_start_us')} to {metadata.get('time_end_us')} us")
events[:10]

In [ ]:
def event_summary(events: np.ndarray, sensor_shape: tuple[int, int], top_n: int = 20) -> dict[str, object]:
    if events.size == 0:
        return {
            "event_count": 0,
            "duration_us": 0,
            "positive_events": 0,
            "negative_events": 0,
            "top_pixels": [],
        }

    height, width = sensor_shape
    x = events["x"].astype(np.intp, copy=False)
    y = events["y"].astype(np.intp, copy=False)
    in_bounds = (x >= 0) & (x < width) & (y >= 0) & (y < height)
    flat_pixels = np.ravel_multi_index((y[in_bounds], x[in_bounds]), sensor_shape)
    counts = np.bincount(flat_pixels, minlength=height * width)
    active_count = min(top_n, int(np.count_nonzero(counts)))
    top_flat = np.argpartition(counts, -active_count)[-active_count:] if active_count else np.array([], dtype=np.intp)
    top_flat = top_flat[np.argsort(counts[top_flat])[::-1]]
    top_y, top_x = np.unravel_index(top_flat, sensor_shape)

    return {
        "event_count": int(events.size),
        "time_start_us": int(events["t"].min()),
        "time_end_us": int(events["t"].max()),
        "duration_us": int(events["t"].max() - events["t"].min()),
        "positive_events": int(np.count_nonzero(events["p"] > 0)),
        "negative_events": int(np.count_nonzero(events["p"] <= 0)),
        "top_pixels": [
            {"x": int(px), "y": int(py), "event_count": int(count)}
            for px, py, count in zip(top_x, top_y, counts[top_flat], strict=True)
        ],
    }


summary = event_summary(events, sensor_shape)
for key, value in summary.items():
    if key != "top_pixels":
        print(f"{key}: {value}")

summary["top_pixels"][:20]

In [ ]:
def events_to_frame(
    events: np.ndarray,
    sensor_shape: tuple[int, int],
    start_us: int,
    end_us: int,
    polarity: str = "all",
) -> np.ndarray:
    if end_us <= start_us:
        raise ValueError("end_us must be greater than start_us")

    mask = (events["t"] >= start_us) & (events["t"] < end_us)
    if polarity == "positive":
        mask &= events["p"] > 0
    elif polarity == "negative":
        mask &= events["p"] <= 0
    elif polarity != "all":
        raise ValueError("polarity must be all, positive, or negative")

    frame = np.zeros(sensor_shape, dtype=np.uint32)
    selected = events[mask]
    if selected.size == 0:
        return frame

    height, width = sensor_shape
    x = selected["x"].astype(np.intp, copy=False)
    y = selected["y"].astype(np.intp, copy=False)
    in_bounds = (x >= 0) & (x < width) & (y >= 0) & (y < height)
    np.add.at(frame, (y[in_bounds], x[in_bounds]), 1)
    return frame


default_start = int(metadata.get("time_start_us") or 0)
default_end = min(default_start + 2_000_000, int(metadata.get("time_end_us") or default_start + 2_000_000))
start_box = widgets.IntText(value=default_start, description="Start us")
end_box = widgets.IntText(value=default_end, description="End us")
polarity_picker = widgets.Dropdown(options=["all", "positive", "negative"], value="all", description="Polarity")
render_button = widgets.Button(description="Render frame", button_style="primary")
frame_output = widgets.Output()


def render_frame(_: object | None = None) -> None:
    with frame_output:
        frame_output.clear_output(wait=True)
        frame = events_to_frame(events, sensor_shape, start_box.value, end_box.value, polarity_picker.value)
        print(f"Events in frame: {int(frame.sum()):,}")
        fig, ax = plt.subplots(figsize=(8, 5))
        image = ax.imshow(frame, cmap="magma", origin="upper")
        ax.set_title(f"{start_box.value:,} to {end_box.value:,} us ({polarity_picker.value})")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(image, ax=ax, label="event count")
        plt.show()


render_button.on_click(render_frame)
display(widgets.HBox([start_box, end_box, polarity_picker, render_button]))
display(frame_output)
render_frame()

In [ ]:
def statistics_markdown_path(array_path: Path) -> Path:
    return array_path.with_name(f"{array_path.stem}_statistics.md")


def build_statistics_markdown() -> str:
    array_path = selected_array_path()
    frame = events_to_frame(events, sensor_shape, start_box.value, end_box.value, polarity_picker.value)
    lines = [
        f"# Event Statistics: {array_path.name}",
        "",
        "## Source",
        "",
        f"- Array file: `{array_path}`",
        f"- Raw file: `{metadata.get('raw_path', 'unknown')}`",
        f"- Sensor shape: `{sensor_shape}` as `image[y, x]`",
        "",
        "## Event Summary",
        "",
        f"- Events: {summary['event_count']:,}",
        f"- Time start: {summary.get('time_start_us', 0):,} us",
        f"- Time end: {summary.get('time_end_us', 0):,} us",
        f"- Duration: {summary.get('duration_us', 0):,} us",
        f"- Positive events: {summary['positive_events']:,}",
        f"- Negative events: {summary['negative_events']:,}",
        "",
        "## Frame Window",
        "",
        f"- Start: {start_box.value:,} us",
        f"- End: {end_box.value:,} us",
        f"- Polarity: `{polarity_picker.value}`",
        f"- Events in frame: {int(frame.sum()):,}",
        "",
        "## Most Active Pixels",
        "",
        "| Rank | x | y | Event Count |",
        "|---:|---:|---:|---:|",
    ]
    for rank, pixel in enumerate(summary["top_pixels"], start=1):
        lines.append(
            f"| {rank} | {pixel['x']} | {pixel['y']} | {pixel['event_count']:,} |"
        )
    lines.append("")
    return "\n".join(lines)


statistics_path = statistics_markdown_path(selected_array_path())
statistics_path.write_text(build_statistics_markdown(), encoding="utf-8")
print(f"Wrote {statistics_path}")